# Tutorial 1: Data Preparation

This tutorial covers loading, preprocessing, and saving multimodal data for deepSCENIC.

## Setup

First, let's import the required packages:

In [2]:
import anndata as ad
import scanpy as sc

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

deepSCENIC version: 0.1.0


## Example Dataset: MM_lines (Melanoma Cell Lines)

This tutorial uses the MM_lines dataset from the [SCENIC+ paper](https://www.nature.com/articles/s41592-023-01938-4) {cite:p}`bravo2023scenicplus`. This dataset contains scRNA-seq and scATAC-seq data from nine human melanoma cell lines representing different melanoma states (melanocytic, mesenchymal, and intermediate).

### Data Availability

The raw data is publicly available from multiple sources:

| Source | Description | Link |
|--------|-------------|------|
| **GEO** | Raw sequencing data | [GSE134432](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE134432) |
| **SCope** | Pre-processed .loom files | [Wouters_Human_Melanoma](http://scope.aertslab.org/#/Wouters_Human_Melanoma) |
| **UCSC Cell Browser** | Interactive explorer | [melanoma.cells.ucsc.edu](https://melanoma.cells.ucsc.edu) |

```{note}
You can process the raw data from GEO yourself, or simply follow along with your own multiomics dataset.
```

In [5]:
import os

data_dir = "./data/MM_lines/"
assert os.path.exists(data_dir), "dir not found"

## scRNA and scATAC Preprocessing

deepSCENIC combines RNA expression data, chromatin accessibility data, and genomic sequences to learn gene regulatory networks at the single-cell level. deepSCENIC follows the `scverse` convention for data formatting, and therefore we expect that your data is already preprocessed into the `anndata.AnnData` format. We will further preprocess these data modalities separately and later combine them into one `mudata.MuData` object, which is the format that deepSCENIC uses to run both training and inference.

### scRNA-seq Preprocessing

Load the gene expression matrix:

In [6]:
# Load scRNA-seq data
adata_rna = ad.read_h5ad(f"{data_dir}/ad_rna_raw.h5ad")
print(f"RNA data: {adata_rna.n_obs} cells x {adata_rna.n_vars} genes")

RNA data: 936 cells x 17085 genes


We perform some basic filtering and normalization using standard `scanpy` functionality.

In [7]:
# Standard scRNA-seq preprocessing with scanpy
# (deepSCENIC expects log-normalized RNA data)
sc.pp.filter_genes(adata_rna, min_cells=10)
sc.pp.normalize_total(adata_rna)
sc.pp.log1p(adata_rna)

Additionally, we remove zero-variance genes (if there are any). These are genes that have the same constant expression across all cells (e.g. 0), which means they'll likely be completely uninformative for training.

In [8]:
ds.pp.remove_zero_variance_genes(adata_rna)

In [9]:
adata_rna

AnnData object with n_obs × n_vars = 936 × 15684
    var: 'n_cells'
    uns: 'log1p'

#### Fetch External Resources

Before further preprocessing, we need to fetch transcription factor lists and gene annotations from external databases.

In [10]:
# Fetch TF list from SCENIC+ resources
tfs = ds.fetch_tf_collection(species="human")  # also available: "mouse", "fly"
print(f"Found {len(tfs)} transcription factors")
tfs[:10]

Found 1892 transcription factors


['ZNF354C',
 'KLF12',
 'ZNF143',
 'ZIC2',
 'ZNF274',
 'SP2',
 'ZBTB7A',
 'BCL6B',
 'ZBTB49',
 'ZIC1']

In [11]:
# Fetch gene annotations from Ensembl
annot, chromsizes = ds.fetch_gene_annotation(
    species="hsapiens",
    biomart_host="http://www.ensembl.org",  # default
    transcript_type="protein_coding",
)
print(f"Annotations for {len(annot)} genes")
annot.head()

deepscenic.datasets - INFO - Loading cached gene annotation from /home/VIB.LOCAL/lukas.mahieu/.cache/deepscenic/gene_annot_hsapiens_befba9cb_ucsc_protein_coding.parquet
deepscenic.datasets - INFO - Loaded annotation for 19354 genes from cache


Annotations for 19354 genes


,Chromosome,Start,End,Strand,Transcription_Start_Site,Transcript_type
Gene,,,,,,
MT-ND1,chrM,3307,4262,+,3307,protein_coding
MT-ND2,chrM,4470,5511,+,4470,protein_coding
MT-CO1,chrM,5904,7445,+,5904,protein_coding
MT-CO2,chrM,7586,8269,+,7586,protein_coding
MT-ATP8,chrM,8366,8572,+,8366,protein_coding


#### Mark Transcription Factors

deepSCENIC models gene regulation through transcription factors (TFs). We need to identify which genes in our dataset are TFs so the model knows which genes can act as regulators. This adds an `is_tf` column to `adata_rna.var` and stores the TF order in `adata_rna.uns['tf_order']`.

In [12]:
# Mark TFs in the RNA data
ds.pp.mark_tfs(adata_rna, tfs)
print(f"Marked {adata_rna.var['is_tf'].sum()} TFs in the dataset")

Marked 1463 TFs in the dataset


#### Add Gene Annotations

Gene annotations (chromosome positions) are needed for two purposes:
1. Computing the region-to-gene search space (which regions can regulate which genes)
2. Splitting features by chromosome for model evaluation

This adds `chromosome` and `tss` (transcription start site) columns to `adata_rna.var`.

In [13]:
# Add gene annotations (chromosome, TSS)
ds.pp.add_gene_annotation(adata_rna, annot)
adata_rna.var[["chromosome", "tss", "is_tf"]].head()

deepscenic.pp - INFO - Added ['chromosome', 'tss'] to 12141 / 15684 genes
deepscenic.pp - INFO -   (3543 genes not found in annotation)


,chromosome,tss,is_tf
A1BG,chr19,58353492,False
A1BG-AS1,<NA>,<NA>,False
A2M,chr12,9115919,False
A2M-AS1,<NA>,<NA>,False
A4GALT,chr22,42694986,False


### scATAC-seq Preprocessing

Load the chromatin accessibility matrix. This is required for training a deepSCENIC model, but not required for running inference or gene perturbation analyses using an already-trained model.

```{note}
We strongly recommend using **imputed** accessibility values (instead of raw) from [pycisTopic](https://pycistopic.readthedocs.io/en/latest/). The imputed matrix provides continuous accessibility values that are better suited for the VAE architecture than sparse binary fragment counts.
```

In [15]:
# Load scATAC-seq data (imputed accessibility from pycisTopic)
adata_atac = sc.read_h5ad(f"{data_dir}/ad_atac_raw.h5ad")
print(f"ATAC data: {adata_atac.n_obs} cells x {adata_atac.n_vars} regions")

ATAC data: 936 cells x 288643 regions


#### Mark Differentially Accessible Regions (Optional)

Differentially Accessible Regions (DARs) are cell-type-specific regulatory regions identified through differential accessibility analysis. Marking DARs allows the model to upweight these biologically important regions during training.

```{note}
This step is **optional**. If you have DAR results from (for example) pycisTopic, you can load them here. Otherwise, skip this step.
```

In [16]:
# Optional: Mark DARs if you have pycisTopic differential accessibility results
# ds.pp.mark_dars(adata_atac, dar_dir="path/to/DARs/")

# Alternatively, provide a dictionary mapping cell types to region lists:
# dar_dict = {"CellType1": ["chr1:1000-2000", ...], "CellType2": [...]}
# ds.pp.mark_dars(adata_atac, dar_dict=dar_dict)

## Create MuData

deepSCENIC uses MuData to store multimodal data in a single object. The {func}`~deepscenic.pp.create_mudata` function combines the RNA and ATAC modalities and automatically parses region coordinates from the ATAC `var_names`. We expect the same cells in both data modalities. 

```{note}
The RNA and ATAC datasets must have identical cell barcodes (matching `obs_names`).
```

In [17]:
# Create MuData from RNA and ATAC
mdata = ds.pp.create_mudata(rna=adata_rna, atac=adata_atac)
mdata

MuData object with n_obs × n_vars = 936 × 304327
  2 modalities
    rna:	936 x 15684
      var:	'n_cells', 'is_tf', 'chromosome', 'tss'
      uns:	'log1p'
    atac:	936 x 288643
      var:	'chromosome', 'start', 'end'

### Add Cell Metadata (Optional)

If you have cell-level annotations (cell type, sample ID, batch, etc.), you can add them to `mdata.obs`. These annotations will be saved with the MuData and can be used later for visualization and analysis.

```{note}
Any columns you add to `mdata.obs` will be preserved when saving and loading the data. This is useful for coloring plots by cell type, grouping cells in analyses, or filtering by experimental conditions.
```

In [19]:
# Example: Load cell metadata from a CSV file
import pandas as pd

metadata = pd.read_csv(data_dir + "cells_metadata.csv", index_col=0)
print(f"Metadata columns: {list(metadata.columns)}")

# Add columns to MuData observations
mdata.obs["cell_line"] = mdata.obs_names.map(metadata["MMline"])
mdata.obs["cell_state"] = mdata.obs_names.map(metadata["lineState"])

# Verify the annotations were added
mdata.obs[["cell_line", "cell_state"]].head()

Metadata columns: ['MMline', 'lineState']


,cell_line,cell_state
MM001_0,MM001,MEL
MM001_1,MM001,MEL
MM001_2,MM001,MEL
MM001_3,MM001,MEL
MM001_4,MM001,MEL


## MuData Processing

Now we apply the preprocessing steps that operate on the combined MuData object.

### Compute Region-to-Gene Search Space

The region-to-gene (R2G) search space defines which genomic regions can potentially regulate which genes. This is based on genomic distance: regions closer to a gene's transcription start site (TSS) are more likely to regulate that gene.

The function computes a distance-weighted penalty matrix where:
- `max_distance`: Maximum distance from TSS to consider (default: 1Mb)
- `sigma`: Gaussian decay parameter (default: 100kb) - regions closer to TSS get lower penalty

**Important** Since we need to know the TSS per gene to deduce region-gene links we filter out genes without TSS annotation in this step. 

In [20]:
# Compute r2g penalty matrix (distance-based)
ds.pp.compute_r2g_penalty(
    mdata,
    max_distance=1_000_000,  # 1Mb
    sigma=100_000,  # 100kb Gaussian sigma for decay penalty
)

Computing R2G matrix: 100%|██████████| 24/24 [00:08<00:00,  2.91it/s]
deepscenic.pp.search_space - INFO - 17 TFs without annotation will have 0 links
deepscenic.pp.search_space - INFO - Removed 3526 non-TF genes without annotation from RNA modality
deepscenic.pp.search_space - INFO - Reordered RNA modality: 1463 TFs first, then 10695 other genes


### Split Data for Training and Evaluation

We split the data into training and test sets at two levels:

1. **Cell split**: Random split of cells (e.g., 80% train, 20% test)
2. **Feature split**: Chromosome-based split of genes/regions

```{important}
The feature split enables evaluation on held-out chromosomes to assess generalization to unseen genomic regions. During the main training phase (Phase 1), **reconstruction loss** is computed on train genes and train regions only, and **TF2rNet/Enformer** (tf2r) is trained on train regions only. The **r2g sparsity loss** is also scoped to train-chromosome links only, matching the reconstruction loss scope.

Note that the full model forward pass reconstructs all genes and regions — the feature split only controls which features contribute to the reconstruction loss and which regions tf2r is learned on.
```

#### Transcription Factor Handling

All genes — including transcription factors (TFs) — are split strictly by chromosome. TF expression is **always** available as encoder input regardless of split (the encoder uses the `is_tf` column, not the `split` column). However, only genes on train chromosomes contribute to the Phase 1 reconstruction loss, and only genes on test chromosomes contribute to the Phase 3 (r2g finetuning) reconstruction loss. This applies equally to TFs and non-TF genes.

In other words:
- **TFs on train chromosomes**: `split="train"`, reconstructed in Phase 1 and Phase 2 finetuning
- **TFs on test chromosomes**: `split="test"`, reconstructed in Phase 3 finetuning only
- **TF encoder input**: always uses all TFs (via `is_tf`), unaffected by the split column

#### Split Structure

The split creates a 2×2 matrix for each modality:

```
                        GENES (by TSS chromosome)
                    ┌─────────────┬─────────────┐
                    │   TRAIN     │    TEST     │
                    │ (other chr) │ (chr7,11,   │
                    │             │  18,19)     │
     ┌──────────────┼─────────────┼─────────────┤
     │    TRAIN     │  MAIN       │ r2g FINETUNE│
CELLS│   (80%)      │  TRAINING   │  (Phase 3)  │
     ├──────────────┼─────────────┼─────────────┤
     │    TEST      │ VALIDATION  │ r2g FINETUNE│
     │   (20%)      │             │  (Phase 2)  │
     └──────────────┴─────────────┴─────────────┘
```

**Phase 1 (Main training) uses:**
- Train cells for training, test cells for validation
- Reconstruction loss on train genes and train regions only
- tf2r learned on train regions only
- r2g sparsity loss on train-chromosome links only
- Best model selected by ATAC reconstruction on test chromosomes (test cells, test regions, test tf2r)

**Phase 2 & 3 (r2g finetuning) uses:**
- Phase 2: Test cells to finetune r2g, train cells for validation (train chromosome features)
- Phase 3: Train cells to finetune r2g, test cells for validation (test chromosome features)
- Only r2g weights are updated; all other parameters are frozen

In [21]:
# Split cells (80% train, 20% test)
ds.pp.split_cells(mdata, test_fraction=0.2, seed=42)
print(f"Cell split: {(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test")

deepscenic.pp - INFO - Split cells: 748 train, 188 test


Cell split: 748 train, 188 test


In [22]:
# Split features by chromosome
ds.pp.split_features_by_chromosome(
    mdata,
    test_chromosomes=["chr16"],
)
print(
    f"Gene split: {(mdata['rna'].var['split'] == 'train').sum()} train, {(mdata['rna'].var['split'] == 'test').sum()} test"
)
print(
    f"Region split: {(mdata['atac'].var['split'] == 'train').sum()} train, {(mdata['atac'].var['split'] == 'test').sum()} test"
)

deepscenic.pp - INFO - 17 genes without chromosome annotation assigned to 'train' split
deepscenic.pp - INFO - Split features by chromosome: 8608 ATAC regions, 528 genes (63 TFs) in test set


Gene split: 11630 train, 528 test
Region split: 280035 train, 8608 test


## Saving and Loading

deepSCENIC stores all preprocessed data in a single `.h5mu` file. This includes both modalities, all annotations, the R2G matrix, and split information.

### Save Preprocessed Data

Use {func}`~deepscenic.write` to save the preprocessed MuData. The function validates the data against the deepSCENIC schema before saving to ensure all required fields are present.

In [23]:
# Save preprocessed MuData
ds.write(mdata, data_dir + "preprocessed_data.h5mu")

deepscenic.io - INFO - Wrote data/MM_lines/preprocessed_data.h5mu: 936 cells


### Load Preprocessed Data

In future sessions, use {func}`~deepscenic.read` to load your preprocessed data. The function validates the schema on load to ensure data integrity.

In [26]:
# Load preprocessed MuData in future sessions
# mdata = ds.read("./data/MM_lines_results/preprocessed_data.h5mu")

## Next Steps

In the next tutorial, we'll train the deepSCENIC model:

- {doc}`02_training`